# Supplementary Classroom Tutorial: Parsing Raw AMPT Output & Particle Rapidity Analysis
=========================================================================================

In this classroom laboratory, we will learn **step-by-step** how to write a raw text file parser in Python to read the output files generated by the **A Multi-Phase Transport (AMPT)** model. We will write our own parser from scratch to inspect the data structure, extract particle momenta, and calculate and compare rapidity ($y$) distributions for different particle species.

### Pedagogical Split:
- **Instructor Demonstration:** Step-by-step walkthrough using the **7.7 GeV** dataset (`../Data/subsets/ampt_7.7_sub100.dat`).
- **Student Hands-on Task:** Replicate the parser and perform the analysis using the **39 GeV** dataset (`../Data/subsets/ampt_39_sub100.dat`), and make comparative physical observations.

---

## The AMPT Output Format (`ampt.dat`)
An AMPT data file is a text file containing consecutive events. Each event starts with an **Event Header** line (11 columns), followed by $N$ **Particle Data** lines (9 columns each), where $N$ is the number of particles in that event.

### 1. Event Header Line Structure (11 columns total):

| Col | Field | Description |
|-----|-------|-------------|
| 1   | `event_id`    | Event number |
| 2   | `test_run`    | Test-particle run index |
| **3**   | **`nparticles`** | **Number of particles in this event** (used to count the particle lines that follow!) |
| 4   | `b`           | Impact parameter (fm) |
| 5–6 | `Npart_proj/targ` | Number of participants in each nucleus |
| 7–10 | elastic/inelastic | Collision counts |
| 11  | `phi_RP`      | Reaction-plane angle |

### 2. Particle Data Line Structure (**9 columns** total):

| Col | Field | Description |
|-----|-------|-------------|
| 1   | `PID`  | PDG particle ID: $\pi^+ = 211$, $\pi^- = -211$, $K^+ = 321$, $p = 2212$, $\pi^0 = 111$, $\gamma = 22$ … |
| 2   | `px`   | x-momentum (GeV/$c$) |
| 3   | `py`   | y-momentum (GeV/$c$) |
| 4   | `pz`   | z-momentum (GeV/$c$, along beam) |
| 5   | `mass` | Rest mass (GeV/$c^2$) |
| 6–9 | `x y z t` | Production spacetime (fm, fm/$c$) — not used for momentum analysis |

> **Key distinction:** Event header lines have **11 columns**; particle lines have **9 columns**.  
> The parser reads `nparticles` from the header so it *never* needs to guess which type a line is — they are always read in the correct sequence.

## Part 1: Instructor Demonstration — Parsing the 7.7 GeV Dataset

We will now write a simple, clean file parser in Python using standard file operations (`open()`, `.readline()`, and `.split()`) to read `../Data/subsets/ampt_7.7_sub100.dat`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Charge lookup for common PDG particle IDs
CHARGE_MAP = {
    211: 1,     # pi+
    -211: -1,   # pi-
    111: 0,     # pi0
    321: 1,     # K+
    -321: -1,   # K-
    311: 0,     # K0
    -311: 0,    # anti-K0
    310: 0,     # K_Short
    130: 0,     # K_Long
    2212: 1,    # proton
    -2212: -1,  # antiproton
    2112: 0,    # neutron
    -2112: 0,   # antineutron
    11: -1,     # electron
    -11: 1,     # positron
    22: 0,      # photon
}


def parse_ampt_file(filepath, max_events=100):
    '''
    Parse an AMPT ampt.dat file into a list of event dicts.

    File layout
    -----------
    Every event starts with exactly ONE Event Header line (11 columns):
        col 0 : event_id
        col 1 : test_run
        col 2 : nparticles   <-- tells us how many particle lines follow
        col 3 : b  (impact parameter, fm)
        col 4-5: Npart_proj, Npart_targ
        col 6-9: elastic/inelastic collision counts
        col 10 : phi_RP (reaction-plane angle)

    Followed by nparticles Particle Data lines (9 columns each):
        col 0 : PID  (PDG code)
        col 1 : px   (GeV/c)
        col 2 : py   (GeV/c)
        col 3 : pz   (GeV/c, along beam axis)
        col 4 : mass (GeV/c^2)
        col 5-8: x, y, z, t  (production spacetime, fm / fm/c)

    KEY: We use nparticles from the header so we NEVER need to guess
    whether a line is a header or particle — headers (11 cols) and
    particles (9 cols) are always read in the correct sequence.
    '''
    events = []
    with open(filepath, 'r') as f:
        for ev_idx in range(max_events):

            # --- Step 1: read the event header line (11 columns) ---
            header_line = f.readline()
            if not header_line:
                break  # end of file

            parts = header_line.split()

            # Guard: event header must have exactly 11 columns.
            # If we see fewer, the file is done or corrupted — stop.
            if len(parts) != 11:
                break

            event_num     = int(parts[0])
            num_particles = int(parts[2])    # col 3 = particle count
            impact_param  = float(parts[3])  # col 4 = impact parameter (fm)

            # --- Step 2: read exactly num_particles particle lines (9 columns each) ---
            particles = []
            for _ in range(num_particles):
                p_line = f.readline()
                if not p_line:
                    break  # unexpected EOF
                p_parts = p_line.split()

                # Guard: particle lines must have exactly 9 columns.
                # An 11-column line here would mean the event structure
                # is misaligned — skip it and warn.
                if len(p_parts) != 9:
                    print(f'WARNING (event {event_num}): expected 9-column particle line,'
                          f' got {len(p_parts)} columns: {p_line.strip()}')
                    continue

                pid  = int(p_parts[0])
                px   = float(p_parts[1])
                py   = float(p_parts[2])
                pz   = float(p_parts[3])
                mass = float(p_parts[4])
                # p_parts[5:9] are x, y, z, t (spacetime coords) — not used here

                particles.append({
                    'pid':  pid,
                    'px':   px,
                    'py':   py,
                    'pz':   pz,
                    'mass': mass,
                })

            events.append({
                'event_num':    event_num,
                'impact_param': impact_param,
                'particles':    particles,
            })
    return events


# Load the 7.7 GeV dataset
file_7_7 = '../Data/subsets/ampt_7.7_sub100.dat'
events_7_7 = parse_ampt_file(file_7_7, max_events=100)
print(f'Successfully parsed {len(events_7_7)} events from 7.7 GeV dataset.')
print(f"Event 1: {len(events_7_7[0]['particles'])} particles, "
      f"b = {events_7_7[0]['impact_param']:.2f} fm")

### Calculating Kinematics and Species Classification

Now let's compute **transverse momentum** $p_T$ and **rapidity** $y$ from the momentum components:

$$p_T = \sqrt{p_x^2 + p_y^2}$$

$$y = \frac{1}{2} \ln\left( \frac{E + p_z}{E - p_z} \right), \quad \text{where } E = \sqrt{p_T^2 + p_z^2 + m_0^2}$$

> **Note:** We use only columns 1–5 of each particle line (PID, px, py, pz, mass). The spacetime columns 6–9 (x, y, z, t) are ignored here.

In [ ]:
def calculate_pt(px, py):
    return np.sqrt(px**2 + py**2)


def calculate_rapidity(px, py, pz, mass):
    pt  = calculate_pt(px, py)
    E   = np.sqrt(pt**2 + pz**2 + mass**2)
    # Guard against numerical issues near E = pz
    denom = np.maximum(E - pz, 1e-15)
    return 0.5 * np.log((E + pz) / denom)


# ── Collect rapidity lists for each species at 7.7 GeV ──
y_pions   = []
y_kaons   = []
y_protons = []
y_charged = []
y_neutral = []

for ev in events_7_7:
    for p in ev['particles']:
        # p contains ONLY physics data (pid, px, py, pz, mass)
        # — event header columns were never stored here
        y       = calculate_rapidity(p['px'], p['py'], p['pz'], p['mass'])
        pid_abs = abs(p['pid'])

        # Species classification by PDG code
        if pid_abs == 211:   # pi+ or pi-
            y_pions.append(y)
        elif pid_abs == 321:  # K+ or K-
            y_kaons.append(y)
        elif pid_abs == 2212: # proton or antiproton
            y_protons.append(y)

        # Charged vs. Neutral (using CHARGE_MAP; None means PID not listed)
        charge = CHARGE_MAP.get(p['pid'], None)
        if charge is not None:
            if charge != 0:
                y_charged.append(y)
            else:
                y_neutral.append(y)

print(f'7.7 GeV: {len(y_pions)} pions, {len(y_kaons)} kaons, '
      f'{len(y_protons)} protons collected from {len(events_7_7)} events.')

# ── Plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

bins = np.linspace(-3.0, 3.0, 30)

# Left panel: identified charged species
ax1.hist(y_pions,   bins=bins, histtype='step', color='blue',  linewidth=1.8,
         label=r'Pions ($\pi^\pm$)')
ax1.hist(y_kaons,   bins=bins, histtype='step', color='green', linewidth=1.8,
         label=r'Kaons ($K^\pm$)')
ax1.hist(y_protons, bins=bins, histtype='step', color='red',   linewidth=1.8,
         label=r'Protons ($p/\bar{p}$)')
ax1.set_xlabel('Rapidity  y', fontsize=12)
ax1.set_ylabel('dN / dy  (counts)', fontsize=12)
ax1.set_title('Identified Charged Particle Rapidity — 7.7 GeV', fontsize=13)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(frameon=True)

# Right panel: all charged vs. neutral
ax2.hist(y_charged, bins=bins, histtype='stepfilled',
         color='#6366f1', alpha=0.4, edgecolor='#4f46e5', linewidth=1.8,
         label='All Charged Particles')
ax2.hist(y_neutral, bins=bins, histtype='step', color='orange', linewidth=1.8,
         label='All Neutral Particles')
ax2.set_xlabel('Rapidity  y', fontsize=12)
ax2.set_ylabel('dN / dy  (counts)', fontsize=12)
ax2.set_title('Charged vs. Neutral Multiplicities — 7.7 GeV', fontsize=13)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(frameon=True)

plt.tight_layout()
plt.show()

## Part 2: Student Hands-on Lab — Parsing the 39 GeV Dataset

Now it is your turn! You will adapt the parser and run the analysis on the higher energy **39 GeV** dataset. This will allow you to see how particle yields and longitudinal shapes change with collision energy.

### Your Tasks:
1. Parse the 39 GeV dataset `../Data/subsets/ampt_39_sub100.dat` for 100 events.
2. Extract $y$ distributions for charged pions, charged kaons, and protons.
3. Extract $y$ distributions for all charged particles vs. neutral particles.
4. Plot the results in a 2-panel chart (just like the demonstration above).
5. **Physical Discussion Questions:**
   - Compare the peak value of $dN/dy$ for pions between 7.7 GeV and 39 GeV. What does this tell you about particle production scaling?
   - Look at the width of the rapidity distributions. How does the width scale with beam energy, and how does this relate to the kinematic beam rapidity $y_{\mathrm{beam}}$?

In [ ]:
# TODO: Define path to the 39 GeV file
file_39 = '../Data/subsets/ampt_39_sub100.dat'

# TODO: Parse the 39 GeV dataset for 100 events


# TODO: Initialize list arrays for pions, kaons, protons, charged, and neutral particles


# TODO: Iterate over events, compute rapidity, and classify particles


# TODO: Plot the identified particles and charged vs. neutral distributions for 39 GeV
# Hint: Use the 7.7 GeV plotting code above as a reference template!


### Student Physical Discussion Responses
*(Write your brief explanations and comparisons here)*